In [1]:

from DQNPlayer_HandBitmap import DQNPlayer_HandBitmap
from Deck import Deck
from Game import GameEnv as Game
from RandomPlayer import RandomPlayer
from DQNPlayer import DQNPlayer
from collections import OrderedDict
from bitarray import bitarray
from Player import Player
import itertools

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

MAX_PLAYS = 100

In [2]:
################### UTILITIES ######################

from Deck import DeckHelper

def pad_msb_fixed(action: bitarray) -> bitarray:
    """
    Aggiunge esattamente 2 bit a 0 all'inizio (MSB) 
    per portare un bitarray di 14 bit a 16 bit (2 byte).
    """
    # Creiamo un bitarray di 2 zeri e lo concateniamo a sinistra
    return bitarray('00') + action

cumulated_rewards_learning_agent = []  # Initialize cumulative rewards for the learning agent

def print_game_state(game: Game, players_action: OrderedDict[Player, bitarray], played_cards: bitarray, rewards: dict[Player, int]):
    TOTAL_CARDS = game.deck.num_cards  # Assuming the deck has a num_cards attribute
    print(f"Current Game State:")
    for player in players_action.keys():
        print(f"Player {player.get_id()} - \t Cards Played: {show_bitarray_in_group(game, players_action[player], TOTAL_CARDS//4)}, Num Cards Played: {sum(players_action[player])}, Score: {rewards[player]}")
    print(f"Played Cards: \t      {played_cards}")
    print("-" * 50)

def show_bitarray_in_group(game: Game, bitarr: bitarray, group_size: int) -> str:
    """Helper function to display a bitarray in groups of a specified size for better readability."""
    groups = []
    
    # Iteriamo sui blocchi (es. di 13 in 13 per i semi)
    for start_idx in range(0, len(bitarr), group_size):
        group_bits = bitarr[start_idx : start_idx + group_size]
        cards_str = []
        
        # Iteriamo all'interno del singolo blocco
        for offset, bit in enumerate(group_bits):
            if bit == 1:
                # Calcoliamo l'indice assoluto della carta nel mazzo
                card_idx = start_idx + offset
                rank = DeckHelper.card_to_rank(game.deck, card_idx)
                cards_str.append(str(rank))
            else:
                cards_str.append('0')
                
        groups.append(' '.join(cards_str))
        
    return '    '.join(groups)


def repeat_and_average(n_runs: int = 5):
    """
    Decoratore che esegue la funzione di esperimento target 'n_runs' volte.
    - Fa la media elemento per elemento delle ricompense (gestendo lunghezze variabili).
    - Raggruppa le storie delle azioni in una lista (poiché i bitarray non sono mediabili).
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            all_rewards = []
            all_actions_runs: OrderedDict[int, list[list[bitarray]]] = OrderedDict() # Raccoglie la cronologia azioni intera di ogni singola run
            # OrderedDict[int, list[list[bitarray]]]]
            for run in range(n_runs):
                print(f"\n{'='*50}\n INIZIO ESECUZIONE SPERIMENTALE RUN {run + 1}/{n_runs}\n{'='*50}")
                
                rewards, actions = func(*args, **kwargs)
                all_rewards.append(rewards)
                all_actions_runs = actions  # Salva la cronologia delle azioni per questa run

            print(f"\n{'='*50}\n CALCOLO DELLA MEDIA DEI RISULTATI SU {n_runs} RUN...\n{'='*50}")

            averaged_rewards = OrderedDict()
            all_actions = OrderedDict()
            players = list(all_rewards[0].keys())
            player_round_action_list : list[list[bitarray]] = []

            for player in players:
                # --- GESTIONE REWARDS ---
                player_reward_lists = [run_rewards[player] for run_rewards in all_rewards]
                padded_rewards = list(itertools.zip_longest(*player_reward_lists, fillvalue=np.nan))
                
                # nanmean somma i reward per quello specifico round in tutte le run e lo divide per il numero di run
                mean_rewards = np.nanmean(padded_rewards, axis=1).tolist()
                averaged_rewards[player] = mean_rewards

            

            return averaged_rewards, all_actions_runs
        
        # Copia manuale dei metadati di base della funzione originale per evitare functools.wraps
        wrapper.__name__ = func.__name__
        wrapper.__doc__ = func.__doc__
        wrapper.__dict__.update(func.__dict__)
        return wrapper
    return decorator
###############################################


In [3]:
################# GRAPHIC UTILITIES ###################
MOVING_WINDOW = 50
def plot_mean_stds(collections, collection_names, title, xlabel='Episodi', ylabel='Reward Totale Accumulato', ylimits: list[float] = [], ylog=False):
    """
    Sovrappone le medie di tutte le collezioni di dati in un unico grafico
    per facilitare il confronto diretto.
    
    Args:
        collezioni_dati: Lista di liste (es. [run1_data, run2_data, ...])
        titolo: Titolo del grafico unico.
        labels: Lista di etichette per la legenda (una per collezione).
    """
    n = len(collections)
        
    plt.figure(figsize=(10, 6))

    plt.rcParams.update({
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'legend.fontsize': 14,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'lines.linewidth': 3,           # linea della media più spessa
        'grid.linewidth': 1.5,          # griglia più spessa
    })
    
    # Definiamo una palette di colori per distinguere le serie
    if n == 2:
        colors = ['#0072B2', '#D55E00']  # Scenario 1
    elif n == 4:
        colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']  # Scenario 2
    elif n == 3:
        colors = ['#0072B2', '#D55E00', "#5b466f"]  # Scenario 3
    else:
        # Fallback alla tua palette originale
        cmap = plt.get_cmap('tab10')
        colors = cmap(np.linspace(0, 1, n))
    
    for i, (data, label) in enumerate(zip(collections, collection_names)):
        smoothed_runs = np.array(data)
        media = np.mean(smoothed_runs, axis=0)
        std = np.std(smoothed_runs, axis=0)
        
        # Plot della media
        plt.plot(media, label=f'{label} (Media)', color=colors[i], linewidth=4)
        
        # Plot della deviazione standard (trasparente)
        plt.fill_between(
            range(len(media)),
            media - std,
            media + std,
            color=colors[i],
            alpha=0.15
        )
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.ylim(ylimits if ylimits else None)
    if ylog:
        plt.yscale('log')
    plt.tight_layout()
    plt.show()
def moving_average(dati, window=10000):
    """Calcola la media mobile per smussare il rumore del grafico."""
    return [np.mean(dati[max(0, i - window + 1):i + 1]) for i in range(len(dati))]


In [4]:
################### EXPERIMENT UTILITIES ###################

@repeat_and_average(n_runs=5)
def run_experiment(
    learning_agent: DQNPlayer, 
    TOTAL_PLAYERS: int, 
    MAX_PLAYS: int, 
    TOTAL_CARDS: int, 
    JACKS: int) -> tuple[OrderedDict[int, list[int]], OrderedDict[int, list[list[bitarray]]]]:
    """Esegue l'esperimento per un numero definito di episodi, tracciando le azioni, i risultati e le ricompense.
    Args:
        game (Game): L'ambiente di gioco.
        players (list[Player]): Lista dei giocatori coinvolti nell'esperimento, incluso l'agente di apprendimento.
        learning_agent (DQNPlayer): L'agente di apprendimento che si desidera monitorare.
        MAX_PLAYS (int): Il numero massimo di episodi da eseguire.
        TOTAL_CARDS (int): Il numero totale di carte nel gioco.
        JACKS (int): Il numero di jolly nel gioco.
    Returns:
        tuple[OrderedDict[Player, list[int]], OrderedDict[Player, list[list[bitarray]]]]: Una tupla contenente i cumuli di ricompense per ogni giocatore e la storia delle azioni per ogni round.
    """
    deck = Deck(num_cards=TOTAL_CARDS, jacks=JACKS)  # Exclude jokers for the deck
    game = Game(deck=deck)
    players: list[Player] = [RandomPlayer(player_id=i, total_cards=TOTAL_CARDS - JACKS) for i in range(TOTAL_PLAYERS - 1)]  # Create 3 random players
    players.append(learning_agent)  # Add the learning agent to the list of players
    rewards_for_player: OrderedDict[int, list[int]] = OrderedDict((int(player.get_id()), []) for player in players)  # Initialize cumulative rewards for each player
    for player in players:
        game.add_player(player)  # Ensure all players are added to the game environment

    actions_history: OrderedDict[int, list[list[bitarray]]] = OrderedDict((int(player.get_id()), []) for player in players)  
    # For each player, for each round in an episode, we will store the action taken as a bitarray in a list. 
    for episode in range(MAX_PLAYS):
        print(f"\n\nEpisode {episode + 1}/{MAX_PLAYS}")
        game.reset()  # Reset the game environment for a new episode
        game.start_game()  # Start the game

        round = 0
        episode = 0
        total_cards_played = bitarray(TOTAL_CARDS)  # Initialize a bitarray to keep track of all played cards in the current episode
        
        while True:
            print(f"Round {episode + 1}")
            [print(f"Player {player.get_id()} hand cards left:\t{sum(player._hand)}") for player in game.players]  # Debug: Print each player's hand before playing

            actions = run_actions(game, players, learning_agent)  # Get actions for all players
            for player in players:
                if len(actions_history[int(player.get_id())]) <= round:
                    actions_history[int(player.get_id())].append([])  # Initialize the list for this round if it doesn't exist
                actions_history[int(player.get_id())][round].append(actions[player])  # Store the action taken by each player in the history

            game_results, _, done, truncated, _ = game.step(actions)
            
            print(f"Played cards: {game_results['played_cards']}")  # Debug: Print the results of the game step
            print(f"total_cards_played before update: {total_cards_played}")  # Debug: Print the total cards played before updating
            total_cards_played |= game_results["played_cards"]  # Update the total cards played
            print_game_state(game, actions, total_cards_played, game_results["rewards"])

            update_players_state(players, actions, game_results)  # Update the state of each player based on the actions and results


            [rewards_for_player[int(player.get_id())].append(game_results["rewards"][player]) for player in players]  # Track cumulative rewards for each player

            if done or truncated:
                print(f"Game Over\n")
                round = 0
                break
            round += 1
            episode += 1
    return rewards_for_player, actions_history  # Return the cumulative rewards for the learning agent after all episodes


def run_actions(game: Game, players: list[Player], learning_agent: DQNPlayer) -> OrderedDict[Player, bitarray]:
    """
    Esegue le azioni dei giocatori e restituisce un dizionario ordinato con le azioni.
    """

    actions = OrderedDict()
    played_cards = 0  # Contatore delle carte giocate in questo round prima del turno corrente

    for player in players:
        # CORREZIONE 1: Confronto diretto per identità di memoria (is)
        # Sostituisce il confronto lento tramite id: 'player.get_id() == learning_agent.get_id()'
        if player is learning_agent:
            learning_agent.num_cards_played = played_cards
            
        # Otteniamo la giocata del giocatore (che leggerà num_cards_played aggiornato)
        action = player.play_cards()
        actions[player] = action
        
        # CORREZIONE 2: Sostituito il lentissimo sum(action) con il C-optimized action.count()
        # 'sum(bitarray)' costringe Python a spacchettare ogni bit in un oggetto Bool/Int e sommarli a livello software.
        # '.count()' della libreria bitarray esegue il calcolo direttamente in C a livello hardware.
        played_cards += action.count()

    return actions


def update_players_state(players: list[Player], actions: OrderedDict[Player, bitarray], game_results: dict):
    """
    Aggiorna lo stato di ogni giocatore in base alle azioni e ai risultati del gioco.
    """
    for player in players:
        player.update_state(
            actions[player],
            game_results["played_cards"], 
            game_results["rewards"][player]
        )






In [5]:
############################# 2 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 2
CARDS_PER_PLAYER = 4
JACKS = 0
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers

In [6]:
# ---------------- FULL DECK SPACE AGENT --------------------- #
FULL_DECK_AGENT_2x4 = DQNPlayer(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

FULL_DECK_rewards_for_player_2x4, FULL_DECK_actions_history_2x4 = run_experiment(
    FULL_DECK_AGENT_2x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #


 INIZIO ESECUZIONE SPERIMENTALE RUN 1/5


Episode 1/100
Player 0 received cards: bitarray('11000101')
Player 1 received cards: bitarray('00111010')
Round 1
Player 0 hand cards left:	4
Player 1 hand cards left:	4
Winning suit: 1 with reward 2
Rewards per player per suit: ['0: [2, 0, 0, 0]', '1: [0, 2, 0, 0]']
Joker players: [0]
Played cards: bitarray('10100001')
total_cards_played before update: bitarray('00000000')
Current Game State:
Player 0 - 	 Cards Played: 1 0    0 0    0 0    0 2, Num Cards Played: 2, Score: 2
Player 1 - 	 Cards Played: 0 0    1 0    0 0    0 0, Num Cards Played: 1, Score: 0
Played Cards: 	      bitarray('10100001')
--------------------------------------------------
Round 2
Player 0 hand cards left:	2
Player 1 hand cards left:	3
Winning suit: 3 with reward 3
Rewards per player per suit: ['0: [0, 0, 0, 1]', '1: [0, 0, 0, 2]']
Played cards: bitarray('00000110')
total_cards_played before update: bitarray('10100001')
Current Game State:
Player 0 - 	 Cards Played: 0 

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ---------------- COMPRESSED SPACE AGENT --------------------- #
COMPRESSED_AGENT_2x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

COMPRESSED_rewards_for_player_2x4, COMPRESSED_actions_history_2x4 = run_experiment(
    COMPRESSED_AGENT_2x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #


 INIZIO ESECUZIONE SPERIMENTALE RUN 1/5


Episode 1/100
Player 0 received cards: bitarray('10101100')
Player 1 received cards: bitarray('01010011')
Round 1
Player 0 hand cards left:	4
Player 1 hand cards left:	4
Winning suit: 3 with reward 3
Rewards per player per suit: ['0: [2, 0, 2, 1]', '1: [0, 0, 1, 2]']
Joker players: [1]
Played cards: bitarray('10011111')
total_cards_played before update: bitarray('00000000')
Current Game State:
Player 0 - 	 Cards Played: 1 0    0 0    1 2    0 0, Num Cards Played: 3, Score: 0
Player 1 - 	 Cards Played: 0 0    0 2    0 0    1 2, Num Cards Played: 3, Score: 3
Played Cards: 	      bitarray('10011111')
--------------------------------------------------
Round 2
Player 1 hand cards left:	1
Player 0 hand cards left:	1
Winning suit: 1 with reward 3
Rewards per player per suit: ['1: [0, 1, 0, 0]', '0: [0, 2, 0, 0]']
Played cards: bitarray('01100000')
total_cards_played before update: bitarray('10011111')
Current Game State:
Player 0 - 	 Cards Played: 0 

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
############################# 3 x 4 EXPERIMENT SETTING #############################
TOTAL_PLAYERS = 3
CARDS_PER_PLAYER = 4
TOTAL_CARDS = TOTAL_PLAYERS * CARDS_PER_PLAYER  # Total cards including jokers
JACKS = 0

In [ ]:
# ---------------- FULL DECK SPACE AGENT --------------------- #
FULL_DECK_AGENT_3x4 = DQNPlayer(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

FULL_DECK_rewards_for_player_3x4, FULL_DECK_actions_history_3x4 = run_experiment(
    FULL_DECK_AGENT_3x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)

# ----------------------------------------------------------- #

In [ ]:
print(f"Length of FULL_DECK_actions_history_3x4: {len(FULL_DECK_actions_history_3x4)}")

In [ ]:
# ---------------- COMPRESSED SPACE AGENT --------------------- #
COMPRESSED_AGENT_3x4 = DQNPlayer_HandBitmap(
    player_id=TOTAL_PLAYERS - 1,  # Assign the last player ID to the learning agent
    total_cards=TOTAL_CARDS, 
    cards_per_player=TOTAL_CARDS // TOTAL_PLAYERS,  # Calculate cards per player based on total cards and players
    discount_factor=0.99,
    batch_size=64,
    SYNC_TARGET_EVERY=400,
    replay_buffer_capacity=MAX_PLAYS
)

COMPRESSED_rewards_for_player_3x4, COMPRESSED_actions_history_3x4 = run_experiment(
    COMPRESSED_AGENT_3x4,
    TOTAL_PLAYERS, MAX_PLAYS, TOTAL_CARDS, JACKS=JACKS
)
# ----------------------------------------------------------- #

In [ ]:
############# PLOT ESTIMATION ERRORS #############
MOVING_WINDOW = 50
FULL_DECK_estimation_errors_2x4, FULL_DECK_target_errors_2x4 = FULL_DECK_AGENT_2x4.get_errors()  # Get the estimation and target errors from the learning agent
FULL_DECK_estimation_errors_3x4, FULL_DECK_target_errors_3x4 = FULL_DECK_AGENT_3x4.get_errors()  # Get the estimation and target errors from the learning agent
COMPRESSED_estimation_errors_2x4, COMPRESSED_target_errors_2x4 = COMPRESSED_AGENT_2x4.get_errors()  # Get the estimation and target errors from the learning agent
COMPRESSED_estimation_errors_3x4, COMPRESSED_target_errors_3x4 = COMPRESSED_AGENT_3x4.get_errors()  # Get the estimation and target errors from the learning agent
plot_mean_stds(
    [
        [moving_average(FULL_DECK_target_errors_2x4, window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_target_errors_2x4, window=MOVING_WINDOW)], 
    ], 
    collection_names=["Full Deck Target Values", "Compressed Target Values",],
    title="Target Values Comparison (2x4)",
    xlabel="Episodes",
    ylabel="Average Target Value",
)  # Passiamo una lista di run, anche se ne abbiamo solo una
plot_mean_stds(
    [
        [moving_average(FULL_DECK_target_errors_3x4, window=MOVING_WINDOW)],
        [moving_average(COMPRESSED_target_errors_3x4, window=MOVING_WINDOW)],
    ], 
    collection_names=["Full Deck Target Values", "Compressed Target Values"],
    title="Target Values Comparison (3x4)",
    xlabel="Episodes",
    ylabel="Average Target Value",
)  # Passiamo una lista di run, anche se ne abbiamo solo una



In [ ]:
MOVING_WINDOW = 1000
FULL_DECK_difference_2x4 = [FULL_DECK_estimation_errors_2x4[i] - FULL_DECK_target_errors_2x4[i] for i in range(len(FULL_DECK_target_errors_2x4))]  # Create a list of indices for the x-axis
COMPRESSED_difference_2x4 = [COMPRESSED_estimation_errors_2x4[i] - COMPRESSED_target_errors_2x4[i] for i in range(len(COMPRESSED_target_errors_2x4))]  # Create a list of indices for the x-axis
FULL_DECK_difference_3x4 = [FULL_DECK_estimation_errors_3x4[i] - FULL_DECK_target_errors_3x4[i] for i in range(len(FULL_DECK_target_errors_3x4))]  # Create a list of indices for the x-axis
COMPRESSED_difference_3x4 = [COMPRESSED_estimation_errors_3x4[i] - COMPRESSED_target_errors_3x4[i] for i in range(len(COMPRESSED_target_errors_3x4))]  # Create a list of indices for the x-axis
plot_mean_stds(
    [
        [moving_average(FULL_DECK_difference_2x4, window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_difference_2x4, window=MOVING_WINDOW)], ], 
    collection_names=["Full Deck error", "Compressed error"],
    title="TD Errors Comparison (2x4)",
    xlabel="Episodes",
    ylabel="Average Error per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una
plot_mean_stds(
    [
        [moving_average(FULL_DECK_difference_3x4, window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_difference_3x4, window=MOVING_WINDOW)], ], 
    collection_names=["Full Deck error", "Compressed error"],
    title="TD Errors Comparison (3x4)",
    xlabel="Episodes",
    ylabel="Average Error per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una

In [ ]:
################ PLOT REWARDS ################
MOVING_WINDOW = 100  # Ad esempio, se MAX_PLAYS è 100, una finestra di 10 episodi per la media mobile
plot_mean_stds(
    [
        [moving_average(FULL_DECK_rewards_for_player_2x4[FULL_DECK_AGENT_2x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_rewards_for_player_2x4[COMPRESSED_AGENT_2x4.get_id()], window=MOVING_WINDOW)], 
        [moving_average(COMPRESSED_rewards_for_player_2x4[0], window=MOVING_WINDOW)],
    ], 
    collection_names=["Full Deck Rewards", "Compressed Rewards","Random Player Rewards"],
    title="Rewards Comparison (2x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una
plot_mean_stds(
    [
        [moving_average(FULL_DECK_rewards_for_player_3x4[FULL_DECK_AGENT_3x4.get_id()], window=MOVING_WINDOW)],
        [moving_average(COMPRESSED_rewards_for_player_3x4[COMPRESSED_AGENT_3x4.get_id()], window=MOVING_WINDOW)],
        [moving_average(COMPRESSED_rewards_for_player_3x4[1], window=MOVING_WINDOW)],
    ], 
    collection_names=["Full Deck Rewards", "Compressed Rewards","Random Player Rewards"],
    title="Rewards Comparison (3x4)",
    xlabel="Episodes",
    ylabel="Average Rewards per Episode",
)  # Passiamo una lista di run, anche se ne abbiamo solo una

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_bar_mean_std(
    data_collections,          # lista di 4 collezioni; ogni collezione è lista di run (liste di valori)
    collection_names,          # lista di 4 nomi
    title,
    xlabel='Numero Run',
    ylabel='Reward medio (ultimi 10 episodi)',
    ylimits=None,
    ylog=False,
    n_last=MAX_PLAYS                  # quanti ultimi valori considerare per ogni run
):
    """
    Per ogni run di ogni collezione calcola la media e la deviazione standard degli ultimi n_last episodi.
    Poi disegna un grafico a barre raggruppate: in posizione i (run i) compaiono le barre di tutte le collezioni
    che hanno almeno i run. Le collezioni con meno run hanno barra 0.
    """
    n_coll = len(data_collections)
    if len(collection_names) != n_coll:
        collection_names = [f'Collezione {i+1}' for i in range(n_coll)]

    # Determina il numero massimo di run tra le collezioni
    max_runs = max(len(coll) for coll in data_collections)

    # Matrici per medie e deviazioni standard delle run (righe=collezioni, colonne=run)
    means_matrix = np.zeros((n_coll, max_runs))
    stds_matrix = np.zeros((n_coll, max_runs))

    for c, coll in enumerate(data_collections):
        for r, run in enumerate(coll):
            if r >= max_runs:
                break
            # Prendi gli ultimi n_last (o tutti se la run è più corta)
            last = run[-n_last:] if len(run) >= n_last else run
            if len(last) == 0:
                means_matrix[c, r] = 0.0
                stds_matrix[c, r] = 0.0
            else:
                means_matrix[c, r] = np.mean(last)
                stds_matrix[c, r] = np.std(last, ddof=1)   # deviazione standard campionaria della run

    # Plot
    x = np.arange(max_runs)                     # posizioni dei gruppi (run 1, run 2, ...)
    width = 0.8 / n_coll                        # larghezza di ogni barra

    fig, ax = plt.subplots(figsize=(12, 6))
    for c in range(n_coll):
        offset = (c - (n_coll - 1) / 2) * width
        ax.bar(
            x + offset,
            means_matrix[c],
            width,
            yerr=stds_matrix[c],
            label=collection_names[c],
            capsize=5,
            alpha=0.8,
            edgecolor='black'
        )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([str(i+1) for i in range(max_runs)])   # etichette 1,2,3,...
    ax.legend()

    if ylimits is not None:
        ax.set_ylim(ylimits)
    if ylog:
        ax.set_yscale('log')

    plt.tight_layout()
    plt.show()

In [ ]:
def compute_value_action(action: bitarray, TOTAL_CARDS: int) -> int:
    """
    Calcola il valore della giocata basato sulla somma dei rank delle carte associate al bitarray.
    """
    if not isinstance(action, bitarray) or len(action) != TOTAL_CARDS:
        raise ValueError(f"Action must be a bitarray of length {TOTAL_CARDS}. Found length: {len(action)}, Type: {type(action)}")
    cards = DeckHelper.bitarray_to_cards(TOTAL_CARDS, action)
    return sum(DeckHelper.card_to_rank(Deck(num_cards=TOTAL_CARDS), card) for card in cards)

def compute_value_actions_per_round(actions_history: list[list[bitarray]], TOTAL_CARDS: int) -> list[int]:
    """
    Per ogni giocatore, per ogni round, calcola il valore della giocata basato sulla somma dei rank delle carte associate al bitarray.
    """
    values_per_round = []
    for round_actions in actions_history:
        round_values = [compute_value_action(action, TOTAL_CARDS) for action in round_actions]
        values_per_round.append(round_values)
        
        
    return values_per_round



FULL_DECK_actions_values_per_round_2x4 = compute_value_actions_per_round(FULL_DECK_actions_history_2x4[FULL_DECK_AGENT_2x4.get_id()], TOTAL_CARDS=8)
COMPRESSED_actions_values_per_round_2x4 = compute_value_actions_per_round(COMPRESSED_actions_history_2x4[COMPRESSED_AGENT_2x4.get_id()], TOTAL_CARDS=8)
FULL_DECK_actions_values_per_round_3x4 = compute_value_actions_per_round(FULL_DECK_actions_history_3x4[FULL_DECK_AGENT_3x4.get_id()], TOTAL_CARDS=12)
COMPRESSED_actions_values_per_round_3x4 = compute_value_actions_per_round(COMPRESSED_actions_history_3x4[COMPRESSED_AGENT_3x4.get_id()], TOTAL_CARDS=12)

print(f"FULL_DECK_actions_values_per_round_2x4: {len(COMPRESSED_actions_values_per_round_3x4)} rounds, with values: {FULL_DECK_actions_values_per_round_2x4}")
plot_bar_mean_std(
    [
        FULL_DECK_actions_values_per_round_2x4, 
        COMPRESSED_actions_values_per_round_2x4, 
        FULL_DECK_actions_values_per_round_3x4, 
        COMPRESSED_actions_values_per_round_3x4
    ],
    collection_names=["Full Deck Values 2x4", "Compressed Values 2x4", "Full Deck Values 3x4", "Compressed Values 3x4"],
    title="Average Action Values in Last 10 Episodes",
    n_last=MAX_PLAYS,
)